In [29]:
# import time
# import pandas as pd
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from webdriver_manager.chrome import ChromeDriverManager

# def scrape_olympic_medals():
#     print("Initializing Chrome Driver...")
    
#     chrome_options = Options()
#     chrome_options.add_argument("--headless")  
#     chrome_options.add_argument("--no-sandbox")
#     chrome_options.add_argument("--disable-dev-shm-usage")
#     chrome_options.add_argument("--disable-blink-features=AutomationControlled")
#     chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

#     service = Service(ChromeDriverManager().install())
#     driver = webdriver.Chrome(service=service, options=chrome_options)
    
#     url = "https://www.olympics.com/en/olympic-games/paris-2024/medals"
    
#     try:
#         print(f"Navigating to: {url}")
#         driver.get(url)
#         time.sleep(6)  # Give structural scripts plenty of time to build components
        
#         try:
#             cookie_btn = driver.find_element(By.ID, "onetrust-accept-btn-handler")
#             cookie_btn.click()
#             print("Cookie banner dismissed successfully.")
#             time.sleep(2)
#         except:
#             pass  
            
#         print("Waiting for medal table elements to load...")
#         wait = WebDriverWait(driver, 20)
        
#         # Target the individual team/country table blocks specifically 
#         # instead of parent structural wrappers.
#         country_selector = "div[data-testid='medal-standings-row'], [class*='MedalStandingsRow'], div[data-testid='row']"
#         try:
#             wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, country_selector)))
#             rows = driver.find_elements(By.CSS_SELECTOR, country_selector)
#         except:
#             # Fallback to broader list cells if specific ID hooks shifted
#             wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.styles__Row-sc-")))
#             rows = driver.find_elements(By.CSS_SELECTOR, "div.styles__Row-sc-")
            
#         print(f"Successfully located {len(rows)} data rows on page!")
        
#         parsed_data = []
#         for index, row in enumerate(rows):
#             try:
#                 row_text = row.text.strip()
#                 if not row_text:
#                     continue
                
#                 # Split entries by newlines
#                 lines = [line.strip() for line in row_text.split('\n') if line.strip()]
                
#                 # Filter out obvious structural text artifacts
#                 if "Total" in lines or len(lines) < 3:
#                     continue
                    
#                 parsed_data.append(lines)
#             except:
#                 continue
                
#         return parsed_data

#     except Exception as e:
#         print(f"An error occurred during scraping execution: {e}")
#         driver.save_screenshot("scraping_error_page.png")
#         return None
#     finally:
#         print("Closing browser driver session.")
#         driver.quit()

# # Run the fixed engine
# raw_scraped_data = scrape_olympic_medals()

In [30]:
import time
import sys
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. Terminal Initialization logs
print("Initializing headless Chrome browser environment...")
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

print("Installing and configuring Chrome WebDriver...")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

url = "https://en.wikipedia.org/wiki/2024_Summer_Olympics_medal_table"
print(f"Connecting to target URL: {url}")

try:
    driver.get(url)
    print("Waiting for data tables to resolve on DOM...")
    wait = WebDriverWait(driver, 15)
    table_locator = (By.CSS_SELECTOR, "table.wikitable.sortable")
    wait.until(EC.presence_of_element_located(table_locator))
    
    print("Target medal table located. Extracting structural elements...")
    table = driver.find_element(*table_locator)
    rows = table.find_elements(By.TAG_NAME, "tr")
    
    raw_scraped_data = []
    for row in rows:
        cells = row.find_elements(By.XPATH, "./th|./td")
        row_data = [cell.text.strip() for cell in cells if cell.text.strip()]
        if row_data:
            raw_scraped_data.append(row_data)
            
    # Print the specific structural log with row count
    print(f"Successfully located {len(raw_scraped_data)} data rows inside the medal table!")

except Exception as e:
    print(f"Operational error occurred during execution: {str(e)}")
    
finally:
    print("Closing automated browser context safely...")
    driver.quit()

# 2. Final memory verification verification statement
print("--------------------------------------------------")
if 'raw_scraped_data' in locals() and len(raw_scraped_data) > 0:
    print("STATUS: Data successfully saved in memory.")
else:
    print("STATUS: Data not saved in memory.")

Initializing headless Chrome browser environment...
Installing and configuring Chrome WebDriver...
Connecting to target URL: https://en.wikipedia.org/wiki/2024_Summer_Olympics_medal_table
Waiting for data tables to resolve on DOM...
Target medal table located. Extracting structural elements...
Successfully located 94 data rows inside the medal table!
Closing automated browser context safely...
--------------------------------------------------
STATUS: Data successfully saved in memory.


In [31]:
import os
import pandas as pd

# 1. Structure the raw memory list into a dataframe table
headers = raw_scraped_data[0]
data_rows = raw_scraped_data[1:]

# Standardize columns to match the 6 Wikipedia layout elements
processed_rows = [row[:6] for row in data_rows if len(row) >= 5]
df_scraped_2024 = pd.DataFrame(processed_rows, columns=['Rank', 'NOC', 'Gold', 'Silver', 'Bronze', 'Total'])

# 2. Redirect the file path directly to the Downloads folder
downloads_path = os.path.join(os.path.expanduser('~'), 'Downloads')
target_file = os.path.join(downloads_path, 'paris_2024_scraped_dirty.csv')

# 3. Save the dataframe to the file
df_scraped_2024.to_csv(target_file, index=False)

# 4. Print explicit tracking statements for your teacher
print("--- PREVIEW OF SAVED DATA ---")
print(df_scraped_2024.head(5))
print("--------------------------------------------------")
if os.path.exists(target_file):
    print(f"STATUS: Raw file successfully saved to Downloads as 'paris_2024_scraped_dirty.csv'")
else:
    print("STATUS: File save execution failed.")

--- PREVIEW OF SAVED DATA ---
  Rank             NOC Gold Silver Bronze Total
0    1  United States‡   40     44     42   126
1    2           China   40     27     24    91
2    3           Japan   20     12     13    45
3    4       Australia   18     19     16    53
4    5         France*   16     26     22    64
--------------------------------------------------
STATUS: Raw file successfully saved to Downloads as 'paris_2024_scraped_dirty.csv'


In [32]:
import numpy as np

# 1. Identify the 29 rows where 'Total' is missing
mask = df_scraped_2024['Total'].isnull()

# 2. Temporarily convert columns to string type to prevent dtype conflicts
for col in ['Rank', 'NOC', 'Gold', 'Silver', 'Bronze', 'Total']:
    df_scraped_2024[col] = df_scraped_2024[col].astype(str).str.strip()

# 3. Shift the data rightward into the correct columns
df_scraped_2024.loc[mask, 'Total'] = df_scraped_2024.loc[mask, 'Bronze']
df_scraped_2024.loc[mask, 'Bronze'] = df_scraped_2024.loc[mask, 'Silver']
df_scraped_2024.loc[mask, 'Silver'] = df_scraped_2024.loc[mask, 'Gold']
df_scraped_2024.loc[mask, 'Gold'] = df_scraped_2024.loc[mask, 'NOC']

# 4. Put the country name back into the NOC column
df_scraped_2024.loc[mask, 'NOC'] = df_scraped_2024.loc[mask, 'Rank']

# 5. Fix the Rank column for tied countries by carrying down the rank from above
rank_mask = mask & (~df_scraped_2024['NOC'].str.contains('Totals', na=False, case=False))
df_scraped_2024.loc[rank_mask, 'Rank'] = np.nan
df_scraped_2024['Rank'] = df_scraped_2024['Rank'].ffill()

# 6. Convert the medal counts back into clean numeric columns
# (Using errors='coerce' turns non-numeric text like the summary row into NaN safely)
for col in ['Gold', 'Silver', 'Bronze', 'Total']:
    df_scraped_2024[col] = pd.to_numeric(df_scraped_2024[col].str.replace(',', ''), errors='coerce')

In [33]:
df_scraped_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Rank    93 non-null     object
 1   NOC     93 non-null     object
 2   Gold    93 non-null     int64 
 3   Silver  93 non-null     int64 
 4   Bronze  93 non-null     int64 
 5   Total   93 non-null     int64 
dtypes: int64(4), object(2)
memory usage: 4.5+ KB


In [34]:
# count of null values broken down for each individual column
df_scraped_2024.isnull().sum() 

Rank      0
NOC       0
Gold      0
Silver    0
Bronze    0
Total     0
dtype: int64

In [35]:
# To get the total number of null values in the entire DataFrame
df_scraped_2024.isnull().sum().sum()


np.int64(0)

In [38]:
# Keep only rows where NOC is not 0 and not '0'
df_scraped_2024 = df_scraped_2024[(df_scraped_2024['NOC'] != 0) & (df_scraped_2024['NOC'] != '0')]

# Verify the count
print(f"New row count: {len(df_scraped_2024)}")

New row count: 93


In [39]:
# 1. See the final, perfectly cleaned total number of rows and columns
print(f"New DataFrame shape: {df_scraped_2024.shape}")

# 2. Confirm that there are absolutely 0 nulls remaining in the NOC column
print(f"Nulls left in NOC: {df_scraped_2024['NOC'].isnull().sum()}")

# 3. Confirm that the total null count for the ENTIRE dataset is a flat 0
print(f"Total nulls across all columns: {df_scraped_2024.isnull().sum().sum()}")

New DataFrame shape: (93, 6)
Nulls left in NOC: 0
Total nulls across all columns: 0


In [40]:
# Fill all remaining missing medal counts with 0
df_scraped_2024 = df_scraped_2024.fillna(0)

# Double-check the grand total now
df_scraped_2024.isnull().sum().sum()

np.int64(0)

In [41]:
# To check for duplicate rows in your DataFrame
df_scraped_2024.duplicated().sum()

np.int64(0)

In [42]:
# To get the statistical summary of your DataFrame
df_scraped_2024.describe()

,Gold,Silver,Bronze,Total
count,93.000000,93.000000,93.000000,93.000000
mean,7.075269,7.096774,8.279570,22.451613
std,34.446946,34.503643,40.019388,108.841879
min,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,1.000000,2.000000
50%,1.000000,1.000000,2.000000,5.000000
75%,3.000000,3.000000,5.000000,9.000000
max,329.000000,330.000000,385.000000,1044.000000


In [43]:
# To check the exact data type of every column
df_scraped_2024.dtypes

Rank      object
NOC       object
Gold       int64
Silver     int64
Bronze     int64
Total      int64
dtype: object

In [44]:
# Convert all numeric columns to numbers at once
numeric_cols = ['Rank', 'Gold', 'Silver', 'Bronze', 'Total']
df_scraped_2024[numeric_cols] = df_scraped_2024[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Check the updated data types to verify the fix
df_scraped_2024.dtypes

Rank      float64
NOC        object
Gold        int64
Silver      int64
Bronze      int64
Total       int64
dtype: object

In [45]:
# to see exactly how many country names are corrupted
# Show all unique country names to spot the dirty characters
df_scraped_2024['NOC'].unique()

array(['United States‡', 'China', 'Japan', 'Australia', 'France*',
       'Netherlands', 'Great Britain', 'South Korea', 'Italy', 'Germany',
       'New Zealand', 'Canada', 'Uzbekistan', 'Hungary', 'Spain',
       'Sweden', 'Kenya', 'Norway', 'Ireland', 'Brazil', 'Iran',
       'Ukraine', 'Romania‡', 'Georgia', 'Belgium', 'Bulgaria', 'Serbia',
       'Czech Republic', 'Denmark', 'Azerbaijan', 'Croatia', 'Cuba',
       'Bahrain', 'Slovenia', 'Chinese Taipei', 'Austria', 'Hong Kong',
       'Philippines', 'Algeria', 'Indonesia', 'Israel', 'Poland',
       'Kazakhstan', 'Jamaica', 'South Africa', 'Thailand',
       'Individual Neutral Athletes[A][B]', 'Ethiopia', 'Switzerland',
       'Ecuador', 'Portugal', 'Greece', 'Argentina', 'Egypt', 'Tunisia',
       'Botswana', 'Chile', 'Saint Lucia', 'Uganda', 'Dominican Republic',
       'Guatemala', 'Morocco', 'Dominica', 'Pakistan', 'Turkey', 'Mexico',
       'Armenia', 'Colombia', 'Kyrgyzstan', 'North Korea', 'Lithuania',
       'India', 'Mold

In [46]:
# Strip out the '*' and '‡' symbols using a regular expression, then clean whitespace
df_scraped_2024['NOC'] = df_scraped_2024['NOC'].str.replace(r'[\*‡]', '', regex=True).str.strip()

# Verify the fix by showing the top 5 rows
df_scraped_2024['NOC'].head(5)

0    United States
1            China
2            Japan
3        Australia
4           France
Name: NOC, dtype: object

In [47]:
# Strip out bracketed text like [A] or [B] using a regular expression
df_scraped_2024['NOC'] = df_scraped_2024['NOC'].str.replace(r'\[\w+\]', '', regex=True).str.strip()

In [48]:
# Verify it is perfectly clean now
print("Verified row look:")
print(df_scraped_2024[df_scraped_2024['NOC'].str.contains('Individual Neutral Athletes', na=False)]['NOC'])

Verified row look:
46    Individual Neutral Athletes
Name: NOC, dtype: object


In [51]:
from pathlib import Path

# 1. Dynamically locate your computer's home directory and target the Downloads folder
downloads_path = Path.home() / "Downloads" / "paris_2024_cleaned.csv"

# 2. Export the final 77-row DataFrame to that exact path
# index=False ensures you don't save messy row number columns
df_scraped_2024.to_csv(downloads_path, index=False)

print("==================================================")
print(f" SUCCESS: Final pristine file saved to:")
print(f"   {downloads_path}")
print("==================================================")
print(f"Final Row Count Verification: {len(df_scraped_2024)} rows")

 SUCCESS: Final pristine file saved to:
   C:\Users\saba\Downloads\paris_2024_cleaned.csv
Final Row Count Verification: 93 rows
